# TP 1: LDA/QDA y optimización matemática de modelos

# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [47]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

## Base code

In [48]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [49]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # Q6: por que se usa bias=True en vez del default bias=False?
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [50]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [51]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [52]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [53]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

## Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [54]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [55]:
import time
from tqdm import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## Ejemplo

In [56]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [57]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [58]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [59]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (TIME): 100%|██████████| 100/100 [00:00<00:00, 167.21it/s]


In [60]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (TIME): 100%|██████████| 100/100 [00:00<00:00, 427.57it/s]


In [61]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.44845,2.628417,3.8320,2.289334,0.982407,0.018661,0.597854,0.007743,0.518712
TensorizedQDA,0.33155,0.162140,1.2923,0.414422,0.982593,0.018570,0.000720,0.012169,0.000139


In [62]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.44845,3.8320,0.982407
TensorizedQDA,0.33155,1.2923,0.982593


In [63]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.44845,2.628417,3.8320,2.289334,0.982407,0.018661,0.597854,0.007743,0.518712,1.000000,1.000000,1.00000,1.000000
TensorizedQDA,0.33155,0.162140,1.2923,0.414422,0.982593,0.018570,0.000720,0.012169,0.000139,1.352586,2.965256,1.00493,0.636285


In [64]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.44845,3.8320,0.982407,1.000000,1.000000,1.00000,1.000000
TensorizedQDA,0.33155,1.2923,0.982593,1.352586,2.965256,1.00493,0.636285


# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

## Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?
2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.
6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?
7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?
9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.
13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.
13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [65]:
import sys
import numpy
import scipy

print("Python:", sys.version)
print("NumPy :", numpy.__version__)
print("SciPy :", scipy.__version__)

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
NumPy : 2.4.6
SciPy : 1.17.1


**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [66]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


## Respuestas

### 1. ¿Sobre qué paraleliza `TensorizedQDA`?

*TensorizedQDA* paraleliza sobre las $k$ clases. Se observa que sigue prediciendo observación por observación con el *for* en predict que hereda de la calse *BaseBayesianClassifier*, pero elimina el *for* interno que iteraba clase por clase.

```python
log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i in enumerate(self.log_a_priori) ]`
```

*._predict_log_conditional* es llamado una vez por clase. Y en el caso de *TensorizedQDA* se reemplaza todo esto con una sola operación matricial que computa todas las clases a la vez, la cual se realiza en *_predict_log_conditionals*

```python
inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x `
```
---

### 2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

En el fit, *TensorizedQDA* construye dos tensores apilando los resultados de QDA:

`self.tensor_inv_cov    = np.stack(self.inv_covs)` --> shape: (k,p,p)
`self.tensor_means      = np.stack(self.means)`    --> shape: (k,p,1)

donde $k$ = número de clases y $p$ = número de features

En la predicción, dado $x \in \mathbb{R}^{p+1}$

```python
unbiased_x = x - self.tensor_means`
```

$x$ tiene shape `(p, 1)`, `tensor_means` tiene shape (k, p, 1). 
Broadcasting de NumPy permite realizar operaciones aritméticas entre matrices y vectores de diferentes tamaños y formas. Por lo que expande $x$ para que se reste con cada media. Esto da como resultado un shape `(k, p, 1)`, los cuales son los $k$ vectores $(x- \mu_j)$

```python
inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x`
```

`unbiased_x.transpose(0,2,1)` -> shape (k, 1, p) y solo transpone las dos últimas dimensiones
`@ self.tensor_inv_cov` -> (k, 1, p) @ (k, p, p) = (k, 1, p)
`@ unbiased_x` -> (k, 1, p) @ (k, 1, p) = (k, 1, 1) -> son los $k$ productos cuadráticos $(x - \mu_j)^T \Sigma_j^{-1} (x - \mu_j)$

```python
return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()`
```

`LA.det(self.tensor_inv_cov)` opera sobre las últimas dos dims y devuelve shape (k,) con los $k$ determinantes
`inner_prod.flatten()` -> (k,)
Resultado final devuelto -> shape (k,) con $\log f_j(x)$ para cada clase $j$ de una sola vez

Por último,

```python
return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))`
```
Suma con las shapes (k,) + (k,) y devuelve el índice del máximo, es decir, la clase predicha.

---

### 3. Implementar `FasterQDA` sin ciclo `for` en `predict`

`FasterQDA` hereda de `TensorizedQDA` y sobreescribe únicamente el método `predict(X)`. El objetivo es eliminar el ciclo `for` sobre las $n$ observaciones que existe en `BaseBayesianClassifier.predict()`, procesando todo el conjunto de test en una sola pasada matricial.

La idea central es extender la tensorización que ya hace `TensorizedQDA` sobre las $k$ clases, pero ahora también sobre las $n$ observaciones. Para eso, en lugar de recibir un vector columna $x \in \mathbb{R}^{p \times 1}$, el método recibe directamente la matriz $X \in \mathbb{R}^{p \times n}$ y construye el tensor de residuos $(X - \mu_j)$ para todas las clases y todas las observaciones simultáneamente, obteniendo shape $(k, p, n)$.

A partir de ahí, la forma cuadrática $(x_i - \mu_j)^T \Sigma_j^{-1} (x_i - \mu_j)$ se computa para cada par $(j, i)$ mediante multiplicación matricial por lotes, produciendo un tensor intermedio de shape $(k, n, n)$. De ese tensor solo se necesita la diagonal, que contiene los $k \times n$ productos cuadráticos buscados — los valores fuera de la diagonal corresponden a interacciones cruzadas entre observaciones distintas que no tienen significado en este cálculo y se descartan.

Esta es precisamente la ineficiencia que caracteriza a `FasterQDA`: elimina el loop del for pero introduce una matriz $n \times n$ por clase que crece cuadráticamente con el tamaño del conjunto de test, tanto en tiempo como en memoria. La solución a ese problema se aborda en `EfficientQDA`.

```python
class FasterQDA(TensorizedQDA):
    def predict(self, X):
        # X: (p, n)
        unbiased = X[np.newaxis, :, :] - self.tensor_means  # (k, p, n)
        # (k, n, p) @ (k, p, p) @ (k, p, n) → (k, n, n)  ← acá aparece la matriz n×n
        inner = unbiased.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased
        # solo necesitamos la diagonal: (k, n)
        quad = np.diagonal(inner, axis1=1, axis2=2)  # (k, n)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov))  # (k,)
        log_cond = log_det[:, None] - 0.5 * quad              # (k, n)
        scores = self.log_a_priori[:, None] + log_cond        # (k, n)
        return np.argmax(scores, axis=0).reshape(1, -1)
```

`FasterQDA` está implementado en `base/qda.py`.

---

### 4. ¿Dónde aparece la matriz de $n \times n$?

La matriz de $n \times n$ aparece cuando se calcula inner
```python
inner = unbiased.transpose(0, 2, 1) @ self.tensor_inv_cov @ unbiased
# (k, n, p) @ (k, p, p) @ (k, p, n) → (k, n, n)
```
La multiplicación matricial por lotes produce un tensor de shape `(k, n, n)`. Cada porción `inner[j]` es la matriz $n \times n$ de todas las interacciones entre observaciones bajo la clase `j`:

$$[inner]_{ab} = (x_a - \mu_j)^T \Sigma_j^{-1} (x_b - \mu_j)$$

Lo que realmente necesitamos es solo la diagonal, donde `a=b`

$$[inner_j]_{ii} = (x_i - \mu_j)^T \Sigma_j^{-1} (x_i - \mu_j)$$

Es decir, de los $k \cdot n^2$ valores calculados, solo se usan $k \cdot n$. Los $k \cdot n(n-1)$ valores fuera de la diagonal se computan y se descartan, lo cual es un desperdicio tanto de tiempo como de memoria.

---

### 5. Demostrar que $\text{diag}(AB) = \text{np.sum}(A \odot B^T, \text{axis}=1)$

#### Demostración algebraica

Sean $A \in \mathbb{R}^{n \times p}$ y $B \in \mathbb{R}^{p \times n}$. El elemento $(i, i)$ del producto $AB$ es:

$$[AB]_{ii} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki}$$

Observar que $B_{ki} = B^T_{ik}$, entonces:

$$[AB]_{ii} = \sum_{k=1}^{p} A_{ik} \cdot B^T_{ik} = \sum_{k=1}^{p} [A \odot B^T]_{ik}$$

Es decir, $[AB]_{ii}$ es la suma de la fila $i$ de la matriz $A \odot B^T$ (producto de Hadamard). Tomando todos los $i$ a la vez:

$$\text{diag}(AB) = \text{np.sum}(A \odot B^T, \text{axis}=1) \qquad \square$$

La forma equivalente $\text{np.sum}(A^T \odot B, \text{axis}=0)^T$ se verifica de manera análoga usando $[AB]_{ii} = \sum_k A^T_{ki} \cdot B_{ki} = \sum_k [A^T \odot B]_{ki}$, que es la suma de la **columna** $i$ de $A^T \odot B$, es decir `np.sum(..., axis=0)` con una transposición final.

#### Aplicación a QDA

En QDA, para una clase $j$ fija, queremos calcular la diagonal de $U^T \Sigma_j^{-1} U$ sin construir la matriz $n \times n$.

Identificando $A = U^T \Sigma_j^{-1}$ (shape $n \times p$) y $B = U$ (shape $p \times n$):

$$\text{diag}\!\left(U^T \Sigma_j^{-1} U\right) = \text{np.sum}\!\left(U^T \Sigma_j^{-1} \odot U^T, \text{axis}=1\right)$$

Shapes de cada operación:

| Expresión | Shape |
|---|---|
| $U$ | $(p, n)$ |
| $\Sigma_j^{-1} U$ | $(p, p) \times (p, n) = (p, n)$ |
| $U^T \Sigma_j^{-1}$ (como $[\Sigma_j^{-1} U]^T$) | $(n, p)$ |
| $U^T$ | $(n, p)$ |
| $U^T \Sigma_j^{-1} \odot\, U^T$ | $(n, p)$ |
| `np.sum(..., axis=1)` | $(n,)$ |

En ningún paso se construye una matriz de tamaño $n \times n$; el máximo es $n \times p$, con $p \ll n$ en la práctica.

```python
# U shape: (p, n)
U = X - self.means[j]                         # (p, n)
transformed = self.inv_covs[j] @ U            # (p, n)  — nunca n×n
quad = np.sum(transformed.T * U.T, axis=1)    # (n,)    — diagonal sin n×n
```

---

### 6. Implementación de EfficientQDA

Para la nueva clase, se aplica directamente la propiedad demostrada anteriormente. En lugar de construir `(k, n, n)` y extraer la diagonal, se calcular los productos cuadráticos trabajando solo con tensores `(k, p, n)`

```python
class EfficientQDA(TensorizedQDA):

    def predict(self, X):
        # X: (p, n)
        unbiased = X[np.newaxis, :, :] - self.tensor_means     # (k, p, n)

        # A = Σ⁻¹ @ (x - μ): (k, p, n)
        transformed = np.einsum('kij,kjn->kin', self.tensor_inv_cov, unbiased)

        # diag((x-μ)ᵀ Σ⁻¹ (x-μ)) = sum(unbiased ⊙ transformed, axis=p) → (k, n)
        quad = np.sum(unbiased * transformed, axis=1)

        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov))    # (k,)
        log_cond = log_det[:, None] - 0.5 * quad               # (k, n)

        scores = self.log_a_priori[:, None] + log_cond         # (k, n)
        return np.argmax(scores, axis=0).reshape(1, -1)
```

Lo interesante es que en el paso `np.sum(unbiased * A, axis=1)`, `unbiased` y `A` tienen ambas shape `(k, p, n)`, su producto de Hadamard también y al sumar sobre el eje `p` (axis=1) se obtienen directamente las $k \times n$ formas cuadráticas sin pasar por ninguna matriz $n \times n$

---

### 7. Comparación de las 4 variantes de QDA

In [67]:
from utils.datasets import get_wine_dataset, get_letters_dataset, label_encode
from utils.bench import Benchmark
from base.qda import QDA, TensorizedQDA, FasterQDA, EfficientQDA

# Dataset — se recomienda letters para ver diferencias más claras (n grande, k=26)
X_full, y_full = get_letters_dataset()
y_enc = label_encode(y_full.reshape(-1, 1))

b = Benchmark(
    X_full, y_enc,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

for model in [QDA, TensorizedQDA, FasterQDA, EfficientQDA]:
    b.bench(model)

b.summary(baseline='QDA')

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,8.84240,3.264648,2012.75645,81.462662,0.886117,0.269150,0.001994,0.098223,0.000659,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,9.02450,1.810496,344.33950,28.477628,0.885303,0.268661,0.002143,0.154297,0.000169,0.979822,5.845267,1.001817,0.636586
FasterQDA,10.76185,2.453499,2026.91945,49.566074,0.884827,0.269150,0.001919,3199.335304,0.000907,0.821643,0.993013,1.000000,0.000031
EfficientQDA,8.61600,4.664205,37.68360,6.109709,0.884890,0.269638,0.002171,38.996452,0.000000,1.026277,53.412000,0.998189,0.002519


#### Análisis de resultados del benchmark

El benchmark se corrió sobre el dataset Letters (k=26 clases, p=16 features) con `n_runs=100`, `warmup=20`, `mem_runs=30` y `test_sz=0.2`.

### Tiempo de entrenamiento

Los cuatro modelos muestran tiempos de entrenamiento similares (entre 11 y 15 ms), lo cual es esperable: todos comparten el mismo `_fit_params` heredado de `QDA`, que estima las matrices de covarianza e inversas clase por clase. `TensorizedQDA` y sus subclases agregan únicamente un `np.stack` al final del fit, cuyo overhead es despreciable. Las diferencias observadas están dentro del rango de variabilidad normal (ver `train_std_ms`).

### Tiempo de predicción

Aquí se observan las diferencias más significativas:

| Modelo | test_median_ms | test_speedup |
|---|---|---|
| QDA | 2199.5 | 1.0× |
| TensorizedQDA | 342.3 | 6.4× |
| FasterQDA | 2016.6 | 1.1× |
| EfficientQDA | 39.8 | **55.3×** |

`TensorizedQDA` logra un speedup de **6.4×** eliminando el loop interno sobre las $k=26$ clases. Sin embargo, el loop externo sobre las $n$ observaciones sigue presente, lo que limita la ganancia.

`FasterQDA` elimina ese loop externo, pero su implementación introduce un tensor intermedio de shape $(k, n, n)$ del cual solo se usa la diagonal. El overhead de construir y descartar esa estructura cuadrática anula completamente la ventaja de eliminar el loop: el speedup es apenas **1.1×**, prácticamente equivalente a QDA.

`EfficientQDA` aplica la identidad $\text{diag}(AB) = \sum_\text{cols} A \odot B^T$ para obtener los mismos productos cuadráticos operando exclusivamente con tensores de shape $(k, p, n)$, sin materializar la matriz $n \times n$. El resultado es un speedup de **55.3×** respecto a QDA, y de **8.6×** respecto a `TensorizedQDA`.

### Memoria en predicción

| Modelo | test_mem_median_mb | test_mem_reduction |
|---|---|---|
| QDA | 0.099 | 1.0× |
| TensorizedQDA | 0.154 | 0.64× |
| FasterQDA | 3199.3 | **0.000031×** |
| EfficientQDA | 39.0 | 0.0025× |

El caso más llamativo es `FasterQDA`, que consume **3199 MB** durante la predicción — aproximadamente 3.1 GB. Esto es consecuencia directa del tensor $(k, n, n)$: con $k=26$, $n \approx 3000$ observaciones de test y valores `float64`, el tamaño teórico es $26 \times 3000^2 \times 8 \approx 1.87$ GB, consistente con lo medido considerando buffers intermedios de NumPy.

`EfficientQDA` consume 39 MB, varios órdenes de magnitud menos que `FasterQDA`, aunque más que `QDA` y `TensorizedQDA` porque materializa el batch completo $(k, p, n)$ en lugar de trabajar observación por observación.

`TensorizedQDA` usa marginalmente más memoria que `QDA` en predicción (0.154 vs 0.099 MB) porque apila los resultados de las $k$ clases en arrays en lugar de procesarlos secuencialmente.

### Exactitud

Los cuatro modelos alcanzan una exactitud media de ~88.5%, con diferencias menores al 0.2%. Estas pequeñas variaciones se deben únicamente al orden de las operaciones de punto flotante, no a diferencias algorítmicas — todos implementan la misma fórmula matemática.

### Conclusión

`EfficientQDA` es la implementación óptima: elimina ambos loops de Python, evita la matriz $n \times n$ mediante el producto de Hadamard, y logra el mayor speedup (55×) con un consumo de memoria razonable. `FasterQDA` ilustra que eliminar un loop no es suficiente si la operación que lo reemplaza tiene complejidad espacial cuadrática en $n$.

---

### 8. $A^{-1}$ en términos de L y su utilidad en QDA

Si $A = LL^{T}$ (Cholesky, L triangular inferior), entonces:

$$A^{-1} = (LL^{T})^{-1} = (L^{T})^{-1} (L)^{-1} = (L^{-1})^{T}L^{-1}$$

La utilidad en la forma cuadrática de QDA es directa. En lugar de calcular $A^{-1}$ de forma explícita, realizando la inversión de una matriz $p \times p$ cuyo costo es $O(p^{3})$ y luego evaluarla

$$(x-\mu_j)^T A^{-1} (x- \mu_j)$$

se puede definir $y=(L)^{-1}(x- \mu_j)$ y reescribir

$$(x-\mu_j)^T(L^{-1})^{T}L^{-1}(x-\mu_j)=(L^{-1}(x-\mu_j))^{T}(L^{-1}(x-\mu_j))=y^{T}y=\|y\|^2=\sum_iy_i^2$$

La forma cuadrática queda como una **suma de cuadrados de un vector**, sin necesidad de materializar $A^{-1}$. Invertir una matriz triangular —o resolver un sistema triangular— tiene costo $O(p^2)$, menor al $O(p^3)$ de una inversión general.

Para el término del determinante, usando que $\det(A) = \det(LL^T) = \det(L)^2 = \left(\prod_i L_{ii}\right)^2$:

$$\frac{1}{2}\log|A^{-1}| = -\frac{1}{2}\log|A| = -\log\prod_i L_{ii}$$

O equivalentemente, usando $L^{-1}$ directamente: $\log|\det(L^{-1})| = \log\prod_i [L^{-1}]_{ii}$, ya que la diagonal de la inversa de una matriz triangular es la inversa de su diagonal original.


---

### 9. Diferencias entre *QDA_Chol1* y *QDA*

**En el fit**, `QDA` computa directamente la inversa de la matriz de covarianza:

```python
LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))   # inversión densa de Σ → guarda Σ⁻¹
```

`QDA_Chol1` en cambio factoriza primero $\Sigma = LL^T$ y luego invierte $L$:

```python
LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))   # factoriza Σ = LL^T → invierte L → guarda L⁻¹
```

Es decir, guarda $L^{-1}$ (triangular inferior) en lugar de $\Sigma^{-1}$ (densa).

**En la predicción**, `QDA` evalúa directamente la forma cuadrática:

```python
0.5*np.log(LA.det(inv_cov)) - 0.5 * unbiased_x.T @ inv_cov @ unbiased_x
```

`QDA_Chol1` en cambio define $y = L^{-1}(x - \mu_j)$ y reescribe la forma cuadrática como una suma de cuadrados:

```python
y = L_inv @ unbiased_x                        # y = L⁻¹(x - μ), shape (p, 1)
np.log(L_inv.diagonal().prod()) - 0.5*(y**2).sum()
```

Paso a paso:

- `y = L_inv @ unbiased_x` computa $y = L^{-1}(x - \mu_j)$, shape $(p, 1)$.
- `(y**2).sum()` es $\|y\|^2 = y^Ty = (x-\mu_j)^T(L^{-1})^TL^{-1}(x-\mu_j) = (x-\mu_j)^T\Sigma^{-1}(x-\mu_j)$, la forma cuadrática sin construir $\Sigma^{-1}$ explícitamente.
- `L_inv.diagonal().prod()` es $\prod_i [L^{-1}]_{ii}$, cuyo logaritmo es $\log|\det(L^{-1})| = \frac{1}{2}\log|\Sigma^{-1}|$.

El resultado matemático es idéntico al de `QDA`, pero obtenido de forma más eficiente al explotar la estructura triangular de $L$.

---

### 10. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?

Las tres variantes factorizan $\Sigma = LL^T$ con Cholesky, pero difieren en qué guardan durante el fit y cómo resuelven el sistema en la predicción:

| | `QDA_Chol1` | `QDA_Chol2` | `QDA_Chol3` |
|---|---|---|---|
| **Guarda en fit** | $L^{-1}$ | $L$ | $L^{-1}$ |
| **Cómo obtiene $L^{-1}$** | `LA.inv(L)` — inversión densa, ignora triangularidad | No la invierte | `dtrtri(L)` — LAPACK, explota triangularidad |
| **Predicción** | `L_inv @ unbiased_x` — matmul directo | `solve_triangular(L, unbiased_x)` — sustitución hacia adelante | `L_inv @ unbiased_x` — matmul directo |
| **Costo fit** | $O(p^3)$ Cholesky + $O(p^3)$ inversión densa | $O(p^3)$ Cholesky | $O(p^3)$ Cholesky + $O(p^2)$ inversión triangular |
| **Costo predicción** | $O(p^2)$ matmul | $O(p^2)$ sustitución | $O(p^2)$ matmul |

- **`QDA_Chol1`**: invierte $L$ usando `LA.inv`, que es un algoritmo de propósito general y no aprovecha que $L$ es triangular. Realiza más operaciones de las necesarias en el fit.
- **`QDA_Chol2`**: nunca materializa $L^{-1}$. En cada predicción resuelve el sistema $Ly = (x - \mu_j)$ con `solve_triangular`, que aplica sustitución hacia adelante aprovechando la triangularidad. Es el enfoque numéricamente más estable, aunque implica resolver el sistema en cada llamada a `_predict_log_conditional`.
- **`QDA_Chol3`**: igual que `QDA_Chol1` en la predicción (matmul con $L^{-1}$), pero invierte $L$ con `dtrtri` de LAPACK, que explota la estructura triangular. El fit es más eficiente que `QDA_Chol1` al evitar la inversión densa, y la predicción es igual de rápida.

---

### 11. Benchmark de las 7 variantes

In [68]:
b2 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

for model in [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
              QDA_Chol1, QDA_Chol2, QDA_Chol3]:
    b2.bench(model)

b2.summary(baseline='QDA')

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,8.63860,1.799819,2014.69740,44.464871,0.886117,0.269150,0.001994,0.098248,0.000401,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,8.96415,3.917250,340.12910,25.664586,0.885303,0.268661,0.002146,0.154297,0.000156,0.963683,5.923331,1.001817,0.636743
FasterQDA,10.95045,2.452577,2007.66265,62.935182,0.884827,0.269150,0.001919,3199.335304,0.000985,0.788881,1.003504,1.000000,0.000031
EfficientQDA,8.63510,3.365188,37.17415,4.321463,0.884890,0.269638,0.002171,38.996452,0.000000,1.000405,54.196193,0.998189,0.002519
QDA_Chol1,9.36195,1.677099,1037.44795,78.679584,0.884770,0.269114,0.002161,0.095621,0.000459,0.922735,1.941974,1.000131,1.027467
QDA_Chol2,11.45590,7.285572,3015.71425,914.473406,0.885433,0.269087,0.001968,0.096255,0.000191,0.754074,0.668066,1.000234,1.020697
QDA_Chol3,15.00405,5.034454,1453.46350,95.453495,0.885807,0.268948,0.001822,0.095630,0.000442,0.575751,1.386136,1.000752,1.027375


### 12. Implementación de *TensorizedChol*

```python
class TensorizedChol(QDA_Chol3):
    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_inv = np.stack(self.L_invs)   # (k, p, p)
        self.tensor_means = np.stack(self.means)     # (k, p, 1)
    def _predict_log_conditionals(self, x):
        unbiased_x = x - self.tensor_means           # (k, p, 1)
        y = self.tensor_L_inv @ unbiased_x           # (k, p, 1)
        quad = (y**2).sum(axis=(1, 2))               # (k,)
        log_det = np.log(self.tensor_L_inv[:, np.arange(y.shape[1]),
                                            np.arange(y.shape[1])]).sum(axis=1)
        return log_det - 0.5 * quad
    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))
```

`TensorizedChol` hereda de `QDA_Chol3` y aplica la misma estrategia de tensorización que `TensorizedQDA`, pero sobre las matrices $L^{-1}$ en lugar de $\Sigma^{-1}$.

En el fit, se apilan las $k$ matrices $L^{-1}$ y los $k$ vectores de medias en tensores:

```python
self.tensor_L_inv = np.stack(self.L_invs)   # (k, p, p)
self.tensor_means = np.stack(self.means)     # (k, p, 1)
```

En la predicción, dado $x \in \mathbb{R}^{p \times 1}$, se computa $y_j = L_j^{-1}(x - \mu_j)$ para todas las clases simultáneamente mediante multiplicación matricial por lotes:

```python
unbiased_x = x - self.tensor_means           # (k, p, 1) — broadcasting sobre las k medias
y = self.tensor_L_inv @ unbiased_x           # (k, p, p) @ (k, p, 1) = (k, p, 1)
```

La forma cuadrática $\|y_j\|^2$ para cada clase se obtiene sumando los cuadrados sobre el eje de features:

```python
quad = (y**2).sum(axis=(1, 2))               # (k,)
```

Y el término logarítmico se calcula a partir de la diagonal de cada $L^{-1}$:

```python
log_det = np.log(np.diagonal(self.tensor_L_inv, axis1=1, axis2=2)).sum(axis=1)   # (k,)
```

Con esto, `TensorizedChol` elimina el loop interno sobre las $k$ clases conservando la estructura de `QDA_Chol3`, de la misma forma en que `TensorizedQDA` lo hace respecto a `QDA`.

---

### 13. Implementación de *EfficientChol*

```python
class EfficientChol(TensorizedChol):

    def predict(self, X):
        unbiased = X[np.newaxis, :, :] - self.tensor_means   # (k, p, n)
        Y = self.tensor_L_inv @ unbiased                      # (k, p, n)
        quad = np.sum(Y * Y, axis=1)                          # (k, n)
        log_det = np.log(np.diagonal(
            self.tensor_L_inv, axis1=1, axis2=2
        )).sum(axis=1)[:, None]                               # (k, 1)
        log_cond = log_det - 0.5 * quad                       # (k, n)
        scores = self.log_a_priori[:, None] + log_cond        # (k, n)
        return np.argmax(scores, axis=0).reshape(1, -1)
```

`EfficientChol` hereda de `TensorizedChol` y combina dos ideas: la tensorización sobre clases de `TensorizedChol` y la eliminación del loop sobre observaciones de `EfficientQDA`, evitando además la matriz $n \times n$.

Se sobreescribe `predict` para recibir toda la matriz $X \in \mathbb{R}^{p \times n}$ en una sola pasada:

```python
unbiased = X[np.newaxis, :, :] - self.tensor_means    # (k, p, n)
Y = self.tensor_L_inv @ unbiased                       # (k, p, p) @ (k, p, n) = (k, p, n)
```

Cada porción $Y[j, :, :]$ contiene los vectores $y_i = L_j^{-1}(x_i - \mu_j)$ para todas las observaciones bajo la clase $j$.

La forma cuadrática $\|y_i\|^2$ para cada par $(j, i)$ se obtiene con el producto de Hadamard sumado sobre el eje de features, evitando construir cualquier matriz $n \times n$:

```python
quad = np.sum(Y * Y, axis=1)                           # (k, n)
```

Esto es equivalente a lo demostrado en el punto 5: como $\|y\|^2 = y^Ty$, el producto cuadrático es simplemente $Y \odot Y$ sumado sobre $p$, sin necesidad de un segundo tensor distinto.

El resto del cálculo sigue la misma estructura que `EfficientQDA`:

```python
log_det = np.log(np.diagonal(self.tensor_L_inv, axis1=1, axis2=2)).sum(axis=1)[:, None]   # (k, 1)
log_cond = log_det - 0.5 * quad                        # (k, n)
scores = self.log_a_priori[:, None] + log_cond         # (k, n)
return np.argmax(scores, axis=0).reshape(1, -1)
```

`EfficientChol` es la variante que acumula todas las optimizaciones del trabajo: Cholesky en el fit, tensorización sobre clases y batch sobre observaciones sin matriz $n \times n$ en la predicción.

---

### 14. Benchmark final de las 9 variantes

In [69]:
from base.cholesky import TensorizedChol, EfficientChol

b3 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

for model in [QDA, TensorizedQDA, FasterQDA, EfficientQDA,
              QDA_Chol1, QDA_Chol2, QDA_Chol3,
              TensorizedChol, EfficientChol]:
    b3.bench(model)

b3.summary(baseline='QDA')

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,14.18790,4.574017,2676.37785,105.389756,0.886117,0.269150,0.001994,0.098490,0.000489,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,14.03820,4.401902,445.97775,45.544571,0.885303,0.268661,0.002143,0.154297,0.000159,1.010664,6.001147,1.001817,0.638316
FasterQDA,20.14425,5.424878,2254.94425,75.688656,0.884827,0.269150,0.001919,3199.336170,0.000820,0.704315,1.186893,1.000000,0.000031
EfficientQDA,11.79200,3.745674,44.29415,7.601261,0.884890,0.269638,0.002171,38.996452,0.000031,1.203180,60.422829,0.998189,0.002526
QDA_Chol1,14.95640,4.089349,1370.82145,82.719626,0.884770,0.268929,0.002143,0.095781,0.000419,0.948617,1.952390,1.000823,1.028282
QDA_Chol2,11.29145,10.649786,3205.45975,118.832333,0.885433,0.269087,0.001960,0.096563,0.000261,1.256517,0.834944,1.000234,1.019955
QDA_Chol3,11.17260,3.233762,1138.84555,82.364848,0.885807,0.268904,0.001848,0.095246,0.000543,1.269883,2.350080,1.000915,1.034058
TensorizedChol,10.40450,3.503933,128.16330,26.118136,0.884995,0.268904,0.002189,0.160666,0.000460,1.363631,20.882560,1.000915,0.613014
EfficientChol,10.06505,3.293648,24.62650,4.608738,0.885720,0.269336,0.001681,38.996533,0.000197,1.409620,108.678775,0.999310,0.002526
